<a href="https://colab.research.google.com/github/cc-huang-0716/internship_tests/blob/main/negative_news_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.colab
from google.colab import drive
import pandas as pd

# 輸入與查看資料
drive.mount("/content/drive")
negative_news_path = "/content/drive/MyDrive/2025-07-09-negative_news.xlsx"
news=pd.read_excel(negative_news_path)
news.head()
print(news.iterrows())
key_word = [
"洗黑錢","貪污","賄賂","內幕交易","毒品","詐欺","走私","逃稅","操控市場","有組織罪行","舞弊","詐欺","電騙","金融犯罪"
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<generator object DataFrame.iterrows at 0x7ae2fd118e10>


In [ ]:
import pandas as pd
file_path="/content/多標籤標註_補齊語意相近.xlsx"
df = pd.read_excel(file_path)

#print(df.shape[0])
#print(df)

#for col in df.select_dtypes(include='number'):
#  print(f"Sum of column {col}: {df[col].sum()}")

#i = 0
#for index, row in df.iterrows():
#  if row[1:].sum() == 0:
#    i += 1
#print(i)

In [ ]:
# 計算特徵矩陣
#X = df.select_dtypes(include="number")
#print(X)

In [ ]:
# 轉張量
import torch
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.validation import check_is_fitted
import joblib

corpus = df['新聞內容'].astype(str).apply(lambda x: x.strip())

vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)

X = vectorizer.transform(corpus)
Y = df.select_dtypes(include="number")

X_tensor = torch.from_numpy(X.toarray()).float()
Y_tensor = torch.from_numpy(Y.to_numpy()).float()

joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
check_is_fitted(vectorizer)

In [ ]:
import sklearn
from sklearn.model_selection import train_test_split

# 分割訓練集
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor,Y_tensor, test_size = 0.1, random_state=42)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# 建立模型
class MLP_MultiLabel(nn.Module):
    def __init__(self, input_dim,output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 16)
        self.out = nn.Linear(16, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        return F.MultiMarginLoss(self.out(x))

In [ ]:
!pip install DataLoader
from torch.utils.data import TensorDataset, DataLoader

# 訓練模型
model = MLP_MultiLabel(input_dim=X_tensor.shape[1], output_dim=Y_tensor.shape[1])
optim = torch.optim.Adam(model.parameters(), lr=0.001)
cr = nn.BCEWithLogitsLoss()
train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=10, shuffle=True)
for epoch in range(50):
  for x, y in train_loader:
    model.train()
    optim.zero_grad()
    logits = model(x)
    print("Model output shape:", logits.shape)
    print("Label shape:", y.shape)
    loss = cr(logits, y)
    loss.backward()
    optim.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

AttributeError: module 'torch.nn.functional' has no attribute 'MultiMarginLoss'

In [ ]:
# 指標
from sklearn.metrics import classification_report
with torch.no_grad():
  model.eval()
  logits = model(X_test)
  y_pred = (torch.sigmoid(logits) > 0.5).int().numpy()
  y_true = Y_test.numpy()

  report = classification_report(
      y_true, y_pred,
      zero_division=0,
      output_dict=True
  )

  report_df = pd.DataFrame(report).transpose()
  report_df.to_excel("多標籤分類_F1評估報告.xlsx")

In [ ]:
file_path1="/content/多標籤分類_F1評估報告.xlsx"
report = pd.read_excel(file_path1)
print(report)

In [ ]:
torch.save(model, 'truemodel.pth')